# Week 3: Turn your DQN into a safety filter

**Embodied AI Safety project · SAIN Utrecht, Technical AI Safety 2026 Q4**

A *safety filter* lets any controller act freely and steps in only when the robot is about to enter a state it cannot recover from. In this project you build one. You change **one line** of ARENA's DQN so that it learns *how safe a state is* instead of *how much reward it will collect*. Then you use the result to stop reckless controllers from crashing: first a cart on a track, then a car steering through a room.

| Part | What you do | Time |
|---|---|---|
| 1 | Learn which states are safe (a cart on a track) | 45 min |
| 2 | Build the filter | 35 min |
| 3 | Filter a robot car in a room | 45 min |
| 4 | Measure how strong your evidence is | 20 min |
| End | Three short answers and the submission check | 20 min |


**This is individual work.** To hand in: run the whole notebook, check that the last cell prints `PASS` for every part, and share the Colab link with the outputs saved (*Share → Anyone with the link can view*). Make sure to also answer the three questions at the end of this notebook 😊.

**What you need.** A free Colab CPU runtime or a laptop. No GPU is required.

**ARENA.** The notebook uses ARENA's own DQN code (section 2.2.1) exactly as it is. You do not need to have done that section, but if DQN is new to you, read its introduction first: you should know what the *TD target* is.

## Setup

The first cell installs the few packages Colab lacks and downloads the seven ARENA Chapter 2 files this project needs, at a fixed version. The second cell writes this project's starter code (two environments, plots, tests and the final check) next to them. Run both; you do not need to read them now.

In [ ]:
import importlib.metadata
import importlib.util
import os
import subprocess
import sys
import urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# ARENA_materials as of 12 November 2026. This notebook was tested against exactly this version;
# to move to a newer ARENA, change the commit and run the notebook from top to bottom again.
ARENA_COMMIT = "32a7107fdd675235a1df61116be4788516a32d92"
ARENA_FILES = ["gpu_env.py", "gpu_probe.py", "rl_utils.py", "plotly_utils.py", "part1_intro_to_rl/utils.py",
               "part21_dqn/solutions.py", "part21_dqn/tests.py"]

# 1. Install only what is missing
needed = {"torch": "torch", "numpy": "numpy", "matplotlib": "matplotlib", "wandb": "wandb>=0.18", "einops": "einops",
          "jaxtyping": "jaxtyping", "torchinfo": "torchinfo", "gymnasium": "gymnasium>=1.2,<2", "plotly": "plotly",
          "cv2": "opencv-python-headless", "imageio": "imageio", "imageio_ffmpeg": "imageio-ffmpeg",
          "tqdm": "tqdm", "eindex": "git+https://github.com/ARENA-education/eindex.git"}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if "gymnasium>=1.2,<2" not in missing and tuple(map(int, importlib.metadata.version("gymnasium").split(".")[:2])) < (1, 2):
    missing.append("gymnasium>=1.2,<2")
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

# 2. Download ARENA's Chapter 2 code
cwd = Path.cwd()
if "chapter2_rl" in cwd.parts:  # the cell was run before, and we are already inside the exercises folder
    root = Path(*cwd.parts[: cwd.parts.index("chapter2_rl")])
else:
    root = Path("/content") if IN_COLAB else cwd
exercises = root / "chapter2_rl" / "exercises"
for name in ARENA_FILES:
    target = exercises / name
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        url = f"https://raw.githubusercontent.com/ARENA-education/ARENA_materials/{ARENA_COMMIT}/chapter2_rl/exercises/{name}"
        target.write_bytes(urllib.request.urlopen(url).read())
(exercises / "part6_safety_filters").mkdir(exist_ok=True)

# 3. Work from the exercises folder, as ARENA's own notebooks do
os.chdir(exercises)
if str(exercises) not in sys.path:
    sys.path.append(str(exercises))
print("Ready. Working in", exercises)

Ready. Working in /content/chapter2_rl/exercises


In [ ]:
#@title Starter code: run this cell (you do not need to read it)
STARTER = r'''"""Starter code for "Turn your DQN into a safety filter" (extends ARENA Chapter 2). CPU only.

It registers two batched environments, written like ARENA's gpu_env.CartPole, in rl_utils.ENVS so that
ARENA's DQNTrainer can build them:
  "DoubleIntegrator-v0": a cart on a track, with its exact safe set (Parts 1-2);
  "RobotCar-v0":         a car steering through a room with obstacles (Parts 3-4).
Also: a grid solver for the car's safe set, rollouts, videos, plots, tests and the final submission check.
"""
import contextlib
import io
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch as t
from gymnasium import spaces

sys.path.append(str(Path(__file__).resolve().parent.parent))  # chapter2_rl/exercises
from gpu_env import GPUEnv, track_episode_stats  # noqa: E402
from rl_utils import ENVS, render_rollout_grid_html  # noqa: E402

DEVICE = t.device("cpu")  # everything in this project is small enough for the CPU
FAILURE_MARGIN = -1.0  # margin of any failed state; strongly negative so failure survives discounting

# ================================================================ Parts 1-2: a cart on a track
P_MAX, V_MAX, A_MAX, DT = 1.0, 2.0, 1.0, 0.05  # track half-length, speed limit, max accel, step (s)


def margin(obs):
    """l(s) for the cart: distance to the nearest end of the track, FAILURE_MARGIN off it (tensors or arrays)."""
    inside = P_MAX - abs(obs[..., 0])
    return inside * (inside >= 0) + FAILURE_MARGIN * (inside < 0)


class DoubleIntegrator(GPUEnv):
    """Batched cart on a frictionless track: v' = clip(v + a DT), p' = p + v' DT, a in {-1, 0, +1} m/s^2.
    Terminates when |p| > P_MAX; reward = l(s) of the state the action was taken in.
    Done envs auto-reset to states drawn uniformly from the box: coverage matters more than reward."""

    def __init__(self, env_id=None, num_envs=16, seed=0, device=None, max_length=200):
        self.num_envs, self.device, self.MAX_LENGTH = num_envs, t.device(device or DEVICE), max_length
        high = np.array([2 * P_MAX, V_MAX], dtype=np.float32)
        self.observation_space = self.single_observation_space = spaces.Box(-high, high, dtype=np.float32)
        self.action_space = self.single_action_space = spaces.Discrete(3)
        self.accels = t.tensor([-A_MAX, 0.0, A_MAX], device=self.device)
        self.gen = t.Generator(device=self.device).manual_seed(seed)
        self.state = self.prev_state = t.zeros(num_envs, 2, device=self.device)
        self.timestep = t.zeros(num_envs, dtype=t.int, device=self.device)
        self._terminated = t.ones(num_envs, dtype=t.bool, device=self.device)  # first reset() fills every env
        self._truncated = t.zeros(num_envs, dtype=t.bool, device=self.device)

    def reset(self):
        done = self._terminated | self._truncated
        infos = {"final_observation": self.state.clone()}
        u = t.rand(self.num_envs, 2, generator=self.gen, device=self.device)
        self.state = t.where(done.unsqueeze(1), (2 * u - 1) * t.tensor([P_MAX, V_MAX], device=self.device), self.state)
        self.timestep[done] = 0
        return self.state, infos

    def set_state(self, states):
        """Start every env from `states` (num_envs, 2), e.g. for evaluation. Returns the observation."""
        self.state = t.as_tensor(states, dtype=t.float32, device=self.device)
        self.timestep = t.zeros_like(self.timestep)
        self._terminated = t.zeros_like(self._terminated)
        self._truncated = t.zeros_like(self._truncated)
        return self.state

    def terminated(self):
        return self.state[:, 0].abs() > P_MAX

    def reward_function(self, action):
        return margin(self.prev_state)

    def step(self, action):
        self.prev_state = self.state
        v = (self.state[:, 1] + self.accels[action] * DT).clamp(-V_MAX, V_MAX)
        self.state = t.stack((self.state[:, 0] + v * DT, v), dim=1)
        self._terminated = self.terminated()
        self.timestep += 1
        self._truncated = self.timestep >= self.MAX_LENGTH
        reward = self.reward_function(action)
        failed = self._terminated
        self.state, infos = self.reset()
        infos.update(track_episode_stats(self, reward, self._terminated, self._truncated))
        infos["failed"] = failed
        return self.state, reward, self._terminated, self._truncated, infos


def is_safe(p, v):
    """Exact safe set of the cart: brake at full power until stopped, never leaving the track."""
    if abs(p) > P_MAX:
        return False
    while v != 0.0:
        v_new = v - np.sign(v) * A_MAX * DT
        if np.sign(v_new) != np.sign(v):  # stops during this step
            v_new = 0.0
        p += v_new * DT
        if abs(p) > P_MAX:
            return False
        v = v_new
    return True


def exact_safe_set(n=101):
    """Grid over the cart's state box, P and V of shape (n, n) indexed [v, p], with the exact safe set."""
    P, V = np.meshgrid(np.linspace(-P_MAX, P_MAX, n), np.linspace(-V_MAX, V_MAX, n))
    return P, V, np.vectorize(is_safe)(P, V)


def sample_safe_states(n, seed=0):
    """n cart states drawn uniformly from the exact safe set, as a (n, 2) tensor."""
    rng, out = np.random.default_rng(seed), []
    while len(out) < n:
        p, v = rng.uniform(-P_MAX, P_MAX), rng.uniform(-V_MAX, V_MAX)
        if is_safe(p, v):
            out.append((p, v))
    return t.tensor(out, dtype=t.float32)


def q_value_fn(q_net):
    """Wrap a Q-network as value_fn(obs) = max_a Q_safe(obs, a), numpy in and out."""
    device = next(q_net.parameters()).device

    @t.inference_mode()
    def value_fn(obs):
        return q_net(t.as_tensor(np.asarray(obs), dtype=t.float32, device=device)).max(-1).values.cpu().numpy()

    return value_fn


def plot_safe_set(value_fn=None, trajectories=(), n=101, ax=None):
    """Cart: exact safe set (black), continuous-time braking curves (dashed) and, given value_fn, the learned
    value as a heatmap with its zero level set (red). trajectories: list of (states (T, 2), interventions (T,) or None)."""
    P, V, safe = exact_safe_set(n)
    ax = ax if ax is not None else plt.subplots(figsize=(6.5, 5))[1]
    if value_fn is not None:
        val = np.asarray(value_fn(np.stack([P.ravel(), V.ravel()], -1))).reshape(P.shape)
        mesh = ax.pcolormesh(P, V, val, cmap="RdBu", vmin=-1, vmax=1, shading="auto")
        plt.colorbar(mesh, ax=ax, label="learned safety value")
        ax.contour(P, V, val, levels=[0.0], colors="red", linewidths=2)
    ax.contour(P, V, safe.astype(float), levels=[0.5], colors="black", linewidths=2)
    v = np.linspace(0, V_MAX, 200)
    ax.plot(P_MAX - v**2 / (2 * A_MAX), v, "k--", lw=1)
    ax.plot(-P_MAX + v**2 / (2 * A_MAX), -v, "k--", lw=1)
    for states, interventions in trajectories:
        states = np.asarray(states)
        ax.plot(states[:, 0], states[:, 1], color="gray", lw=1)
        if interventions is not None:
            hit = np.asarray(interventions, dtype=bool)
            ax.scatter(states[hit, 0], states[hit, 1], s=12, color="orange", zorder=3)
    ax.set(xlabel="position p (m)", ylabel="velocity v (m/s)", xlim=(-P_MAX, P_MAX), ylim=(-V_MAX, V_MAX),
           title="Safe set: exact (black), learned (red), interventions (orange)")
    return ax


# ================================================================ Parts 3-4: a robot car in a room
CAR_SPEED, CAR_TURN, CAR_RADIUS, CAR_DT = 0.5, 2.0, 0.06, 0.05  # m/s, rad/s, m, s (turning radius 0.25 m)
OBSTACLES = ((-0.15, -0.15, 0.24), (0.45, -0.40, 0.18), (-0.45, 0.40, 0.18))  # x, y, radius
GOAL = (0.55, 0.55, 0.14)  # x, y, radius
CAR_FULL_MARGIN = 0.25  # clearance (m) at which the car's margin reaches 1


def car_clearance(xy):
    """Signed distance (m) from the car's edge to the nearest wall or obstacle; negative means a collision."""
    x, y = xy[..., 0], xy[..., 1]
    d = t.minimum(1 - x.abs(), 1 - y.abs())
    for ox, oy, r in OBSTACLES:
        d = t.minimum(d, ((x - ox) ** 2 + (y - oy) ** 2).sqrt() - r)
    return d - CAR_RADIUS


def car_margin(obs):
    """l(s) for the car: its clearance as a fraction of CAR_FULL_MARGIN (capped at 1), or FAILURE_MARGIN
    once it has hit something. Rescaling keeps safe values well away from zero, like the cart's."""
    c = car_clearance(obs)
    return t.where(c >= 0, (c / CAR_FULL_MARGIN).clamp(max=1.0), t.full_like(c, FAILURE_MARGIN))


class RobotCar(GPUEnv):
    """A car moving at constant speed in a 2 m x 2 m room with round obstacles. It can only steer:
    action 0 = right, 1 = straight, 2 = left. Observation: (x, y, cos heading, sin heading).
    The episode terminates when the car hits a wall or an obstacle.

    task=False (for learning Q_safe): short episodes that restart anywhere in the room; reward = l(s) of the pre-step state.
    task=True  (a driving task, for PPO and for evaluation): starts bottom-left; reward for progress
    towards the goal, +1 on arrival, -crash_penalty on a crash; the episode also ends at the goal."""

    def __init__(self, env_id=None, num_envs=16, seed=0, device=None, task=False, max_length=None, crash_penalty=1.0):
        self.num_envs, self.device = num_envs, t.device(device or DEVICE)
        self.task, self.crash_penalty = task, crash_penalty
        self.MAX_LENGTH = max_length or (300 if task else 25)  # short episodes when learning Q_safe: frequent restarts cover the room
        self.observation_space = self.single_observation_space = spaces.Box(-1.0, 1.0, (4,), dtype=np.float32)
        self.action_space = self.single_action_space = spaces.Discrete(3)
        self.turns = t.tensor([-CAR_TURN, 0.0, CAR_TURN], device=self.device)
        self.gen = t.Generator(device=self.device).manual_seed(seed)
        self.state = self.prev_state = t.zeros(num_envs, 3, device=self.device)  # x, y, heading
        self.timestep = t.zeros(num_envs, dtype=t.int, device=self.device)
        self._terminated = t.ones(num_envs, dtype=t.bool, device=self.device)  # first reset() fills every env
        self._truncated = t.zeros(num_envs, dtype=t.bool, device=self.device)

    def observe(self, state):
        return t.stack((state[:, 0], state[:, 1], state[:, 2].cos(), state[:, 2].sin()), dim=1)

    def reset(self):
        done = self._terminated | self._truncated
        final = self.observe(self.state)
        u = t.rand(self.num_envs, 3, generator=self.gen, device=self.device)
        if self.task:  # bottom-left corner, roughly facing the goal
            fresh = t.stack((-0.9 + 0.3 * u[:, 0], -0.9 + 0.3 * u[:, 1], math.pi / 4 + (u[:, 2] - 0.5)), dim=1)
        else:  # anywhere, any heading
            fresh = t.stack((2 * u[:, 0] - 1, 2 * u[:, 1] - 1, (2 * u[:, 2] - 1) * math.pi), dim=1)
        self.state = t.where(done.unsqueeze(1), fresh, self.state)
        self.timestep[done] = 0
        return self.observe(self.state), {"final_observation": final}

    def set_state(self, states):
        """Start every env from `states` (num_envs, 3) = (x, y, heading). Returns the observation."""
        self.state = t.as_tensor(states, dtype=t.float32, device=self.device)
        self.timestep = t.zeros_like(self.timestep)
        self._terminated = t.zeros_like(self._terminated)
        self._truncated = t.zeros_like(self._truncated)
        return self.observe(self.state)

    def failed(self):
        return car_clearance(self.state) < 0

    def goal_distance(self, state):
        return ((state[:, 0] - GOAL[0]) ** 2 + (state[:, 1] - GOAL[1]) ** 2).sqrt()

    def reached_goal(self):
        return (self.goal_distance(self.state) < GOAL[2]) if self.task else t.zeros_like(self._terminated)

    def terminated(self):
        return self.failed() | self.reached_goal()

    def reward_function(self, action):
        if not self.task:
            return car_margin(self.prev_state)
        progress = self.goal_distance(self.prev_state) - self.goal_distance(self.state)
        return 4 * progress + self.reached_goal().float() - self.crash_penalty * self.failed().float()

    def step(self, action):
        self.prev_state = self.state
        heading = self.state[:, 2] + self.turns[action] * CAR_DT
        heading = (heading + math.pi) % (2 * math.pi) - math.pi
        x = self.state[:, 0] + CAR_SPEED * heading.cos() * CAR_DT
        y = self.state[:, 1] + CAR_SPEED * heading.sin() * CAR_DT
        self.state = t.stack((x, y, heading), dim=1)
        failed, reached = self.failed(), self.reached_goal()
        self._terminated = failed | reached
        self.timestep += 1
        self._truncated = self.timestep >= self.MAX_LENGTH
        reward = self.reward_function(action)
        obs, infos = self.reset()
        infos.update(track_episode_stats(self, reward, self._terminated, self._truncated))
        infos["failed"], infos["reached_goal"] = failed, reached
        return obs, reward, self._terminated, self._truncated, infos

    def draw(self, o, canvas, ox, oy, w, h):
        """Renderer for ARENA's 4x4 grid videos: a top-down view of the room. `o` is (x, y, cos, sin), optionally
        followed by three flags: filter overrode the driver (car drawn orange), crashed (red), reached the goal (green)."""
        import cv2

        o = np.asarray(o, dtype=np.float32).reshape(-1)
        flag = lambda i: o.size > i and o[i] > 0.5
        side = min(w, h) - 8
        x0, y0 = ox + (w - side) // 2, oy + (h - side) // 2
        px = lambda x, y: (int(x0 + (x + 1) / 2 * side), int(y0 + (1 - (y + 1) / 2) * side))
        cv2.rectangle(canvas, (x0, y0), (x0 + side, y0 + side), (60, 60, 60), 1)
        for cx, cy, r in OBSTACLES:
            cv2.circle(canvas, px(cx, cy), int(r / 2 * side), (150, 150, 150), -1)
        cv2.circle(canvas, px(GOAL[0], GOAL[1]), int(GOAL[2] / 2 * side), (60, 170, 60), 2)
        color = (210, 40, 40) if flag(5) else (40, 150, 70) if flag(6) else (235, 140, 20) if flag(4) else (40, 90, 200)
        cv2.circle(canvas, px(o[0], o[1]), max(int(CAR_RADIUS / 2 * side), 3), color, -1)
        cv2.line(canvas, px(o[0], o[1]), px(o[0] + 0.16 * o[2], o[1] + 0.16 * o[3]), color, 2)


class RobotCarTask(RobotCar):
    """The driving task: what ARENA's PPOTrainer builds when you set ENV_DICT["classic-control"] = RobotCarTask."""

    def __init__(self, env_id=None, num_envs=64, seed=0, device=None, **kwargs):
        super().__init__(env_id, num_envs=num_envs, seed=seed, device=device, task=True, **kwargs)


ENVS["DoubleIntegrator-v0"] = DoubleIntegrator
ENVS["RobotCar-v0"] = RobotCar


def go_to_goal(obs):
    """A driver that steers straight at the goal and ignores obstacles. obs (n, 4) -> actions (n,)."""
    gx, gy = GOAL[0] - obs[:, 0], GOAL[1] - obs[:, 1]
    angle = t.atan2(obs[:, 2] * gy - obs[:, 3] * gx, obs[:, 2] * gx + obs[:, 3] * gy)  # > 0: goal is on the left
    return t.where(angle > 0.1, 2, t.where(angle < -0.1, 0, 1))


@t.inference_mode()
def car_grid_value(n_xy=81, n_theta=48, device=None, max_iters=600, tol=1e-4):
    """Reference safety value for the car by dynamic programming on a grid over (x, y, heading):
    V <- min(clearance, max_a V(next state)), with interpolation between grid points. This is "exact"
    reachability up to the grid resolution: seconds on a GPU for these 3 dimensions.
    Returns a dict with the grid axes xs and thetas, and V of shape (n_xy, n_xy, n_theta) indexed [x, y, heading]."""
    dev = t.device(device or DEVICE)
    xs = t.linspace(-1, 1, n_xy, device=dev)
    thetas = t.arange(n_theta, device=dev) * (2 * math.pi / n_theta) - math.pi
    X, Y, TH = t.meshgrid(xs, xs, thetas, indexing="ij")
    clearance = car_clearance(t.stack((X, Y), -1)).flatten()
    corners = []  # per action: flat indices of the 8 grid points around each next state, and their weights
    for turn in (-CAR_TURN, 0.0, CAR_TURN):
        th = TH + turn * CAR_DT
        fx = ((X + CAR_SPEED * th.cos() * CAR_DT + 1) / 2 * (n_xy - 1)).clamp(0, n_xy - 1 - 1e-4)
        fy = ((Y + CAR_SPEED * th.sin() * CAR_DT + 1) / 2 * (n_xy - 1)).clamp(0, n_xy - 1 - 1e-4)
        ft = ((th + math.pi) / (2 * math.pi) * n_theta) % n_theta
        ix, iy, it = fx.floor().long(), fy.floor().long(), ft.floor().long()
        wx, wy, wt = fx - ix, fy - iy, ft - it
        idx, w = [], []
        for dx in (0, 1):
            for dy in (0, 1):
                for dth in (0, 1):
                    idx.append(((ix + dx) * n_xy + (iy + dy)) * n_theta + (it + dth) % n_theta)
                    w.append((wx if dx else 1 - wx) * (wy if dy else 1 - wy) * (wt if dth else 1 - wt))
        corners.append((t.stack(idx, -1).reshape(-1, 8), t.stack(w, -1).reshape(-1, 8)))
    V = clearance.clone()
    for _ in range(max_iters):
        V_new = t.minimum(clearance, t.stack([(V[idx] * w).sum(-1) for idx, w in corners]).max(0).values)
        converged = (V - V_new).abs().max() < tol
        V = V_new
        if converged:
            break
    return {"xs": xs, "thetas": thetas, "V": V.reshape(n_xy, n_xy, n_theta)}


def car_grid_obs(grid):
    """The coordinate observation (x, y, cos, sin) of every grid point, shape (n_xy * n_xy * n_theta, 4)."""
    X, Y, TH = t.meshgrid(grid["xs"], grid["xs"], grid["thetas"], indexing="ij")
    return t.stack((X, Y, TH.cos(), TH.sin()), -1).reshape(-1, 4)


def sample_car_starts(n, grid, min_value=0.05, seed=0, task=True):
    """n car states (x, y, heading) that the grid reference calls safe with some room to spare.
    task=True draws them from the bottom-left start area, roughly facing the goal; otherwise from the whole room."""
    gen = t.Generator().manual_seed(seed)
    xs, thetas, V = grid["xs"].cpu(), grid["thetas"].cpu(), grid["V"].cpu()
    out = []
    while sum(len(o) for o in out) < n:
        u = t.rand(4 * n, 3, generator=gen)
        if task:
            s = t.stack((-0.9 + 0.3 * u[:, 0], -0.9 + 0.3 * u[:, 1], math.pi / 4 + (u[:, 2] - 0.5)), dim=1)
        else:
            s = t.stack((2 * u[:, 0] - 1, 2 * u[:, 1] - 1, (2 * u[:, 2] - 1) * math.pi), dim=1)
        ix = ((s[:, 0] + 1) / 2 * (len(xs) - 1)).round().long()
        iy = ((s[:, 1] + 1) / 2 * (len(xs) - 1)).round().long()
        it = ((s[:, 2] + math.pi) / (2 * math.pi) * len(thetas)).round().long() % len(thetas)
        out.append(s[V[ix, iy, it] > min_value])
    return t.cat(out)[:n]


def plot_car_slice(heading_deg=45.0, grid=None, q_net=None, trajectories=(), ax=None):
    """The room for one heading of the car: obstacles (grey), goal (green), the grid reference's safe-set
    boundary (black) and, given q_net, the learned safety value as a heatmap with its zero level set (red).
    trajectories: list of (states (T, >=2), interventions (T,) or None)."""
    ax = ax if ax is not None else plt.subplots(figsize=(6, 5.4))[1]
    th = math.radians(heading_deg)
    axis = np.linspace(-1, 1, 121)
    X, Y = np.meshgrid(axis, axis)
    if q_net is not None:
        obs = np.stack([X.ravel(), Y.ravel(), np.full(X.size, math.cos(th)), np.full(X.size, math.sin(th))], -1)
        val = q_value_fn(q_net)(obs).reshape(X.shape)
        mesh = ax.pcolormesh(X, Y, val, cmap="RdBu", vmin=-1, vmax=1, shading="auto")
        plt.colorbar(mesh, ax=ax, label="learned safety value")
        ax.contour(X, Y, val, levels=[0.0], colors="red", linewidths=2)
    if grid is not None:
        k = int(round((th + math.pi) / (2 * math.pi) * len(grid["thetas"]))) % len(grid["thetas"])
        xs = grid["xs"].cpu().numpy()
        ax.contour(xs, xs, grid["V"][:, :, k].cpu().numpy().T, levels=[0.0], colors="black", linewidths=2)
    for ox, oy, r in OBSTACLES:
        ax.add_patch(plt.Circle((ox, oy), r, color="0.6", alpha=0.9))
    ax.add_patch(plt.Circle(GOAL[:2], GOAL[2], fill=False, color="green", lw=2))
    ax.arrow(-0.85, -0.85, 0.08 * math.cos(th), 0.08 * math.sin(th), head_width=0.05, color="black")
    for states, interventions in trajectories:
        states = np.asarray(states)
        ax.plot(states[:, 0], states[:, 1], color="tab:blue", lw=1.2)
        if interventions is not None:
            hit = np.asarray(interventions, dtype=bool)
            ax.scatter(states[hit, 0], states[hit, 1], s=10, color="orange", zorder=3)
    ax.set(xlim=(-1, 1), ylim=(-1, 1), aspect="equal", xlabel="x (m)", ylabel="y (m)",
           title=f"Car heading {heading_deg:.0f}\N{DEGREE SIGN}")
    return ax


# ================================================================ rollouts, videos, statistics
@t.inference_mode()
def rollout(env, policy, q_net=None, eps=0.0, init_states=None, safety_filter=None, record=False):
    """One episode in every env, driven by policy(obs) -> actions. With q_net, each action first goes through
    safety_filter(q_net(obs), action, eps). Returns a dict with failed and reached_goal (num_envs,) bool and
    intervention_rate (the fraction of steps where the filter changed the action). With record=True it also has
    states (T+1, num_envs, obs_dim) and, per step, interventions, failed_so_far and reached_so_far (T+1, num_envs)."""
    obs = env.set_state(init_states) if init_states is not None else env.reset()[0]
    n = env.num_envs
    active = t.ones(n, dtype=t.bool)
    failed, reached = t.zeros_like(active), t.zeros_like(active)
    steps, hits = t.zeros(n), t.zeros(n)
    states, flags, failed_t, reached_t = [obs.clone()], [], [failed.clone()], [reached.clone()]
    for _ in range(env.MAX_LENGTH):
        a_nom = policy(obs)
        a = a_nom if q_net is None else safety_filter(q_net(obs), a_nom, eps)
        hit = (a != a_nom) & active
        steps, hits = steps + active.float(), hits + hit.float()
        obs, _, term, trunc, infos = env.step(a)
        done = term | trunc
        failed |= infos.get("failed", term) & active
        reached |= infos.get("reached_goal", t.zeros_like(term)) & active
        if record:  # keep each env's own episode: its last state when it ends, frozen afterwards
            now = t.where(done.unsqueeze(1), infos["final_observation"], obs)
            states.append(t.where(active.unsqueeze(1), now, states[-1]))
            flags.append(hit)
            failed_t.append(failed.clone())
            reached_t.append(reached.clone())
        active &= ~done
        if not active.any():
            break
    out = {"failed": failed, "reached_goal": reached, "intervention_rate": (hits.sum() / steps.sum()).item()}
    if record:
        out.update(states=t.stack(states), interventions=t.stack(flags + [t.zeros_like(active)]),
                   failed_so_far=t.stack(failed_t), reached_so_far=t.stack(reached_t))
    return out


def rollout_video(env, result, fps=12):
    """A 4x4 grid video (ARENA style) of the first 16 cars of a recorded rollout.
    Blue: the driver is in charge. Orange: the filter is overriding it. Red: crashed. Green: reached the goal."""
    extra = t.stack([result["interventions"], result["failed_so_far"], result["reached_so_far"]], -1).float()
    frames = t.cat([result["states"], extra], -1).transpose(0, 1)  # (num_envs, T+1, 7)
    frames = t.cat([frames, frames[:, -1:].expand(-1, fps, -1)], 1)  # hold the last frame for a second
    return render_rollout_grid_html(frames, env.draw, fps=fps, cell_w=160, cell_h=160, max_frames=frames.shape[1])


def wilson_interval(k, n, z=1.96):
    """95% Wilson interval for a rate of k events in n trials, e.g. episodes with a violation."""
    if n == 0:
        return 0.0, 1.0
    p, z2 = k / n, z**2
    centre = (p + z2 / (2 * n)) / (1 + z2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z2 / (4 * n**2)) / (1 + z2 / n)
    return max(0.0, centre - half), min(1.0, centre + half)


def report(name, failed, extra=""):
    """Print a failure count with its 95% interval."""
    k, n = int(failed.sum()), len(failed)
    lo, hi = wilson_interval(k, n)
    print(f"{name}: {k}/{n} episodes fail ({lo:.2%}-{hi:.2%}){extra}")


# ================================================================ tests
def test_safety_target(safety_target):
    """safety_target(margin, next_q_max, next_margin, terminated, gamma) -> y, tensors of shape (batch,).
    y = (1 - gamma) l(s) + gamma min(l(s), V'), with V' = max_a' Q(s', a'), or l(s') if terminated."""
    m = t.tensor([0.5, 0.5, 0.5])
    q = t.tensor([0.2, 0.8, 100.0])  # 100 must be ignored: that transition terminated
    m_next = t.tensor([0.3, 0.3, FAILURE_MARGIN])
    done = t.tensor([False, False, True])
    t.testing.assert_close(safety_target(m, q, m_next, done, 0.9), t.tensor([0.23, 0.5, -0.85]), atol=1e-5, rtol=0)
    print("All tests in `test_safety_target` passed!")


def test_safety_filter(safety_filter):
    """safety_filter(q_values, a_nom, eps) -> actions: keep a_nom where Q_safe(s, a_nom) >= eps,
    otherwise argmax_a Q_safe(s, a). q_values has shape (batch, n_actions)."""
    q = t.tensor([[0.3, -0.2, 0.1], [0.3, 0.25, 0.1], [-0.5, -0.4, -0.6]])
    t.testing.assert_close(safety_filter(q, t.tensor([1, 1, 2]), 0.05), t.tensor([0, 1, 1]))
    print("All tests in `test_safety_filter` passed!")


def test_safe_set_iou(value_fn, threshold=0.85, n=101):
    """Cart: intersection over union between the learned safe set {value >= 0} and the exact one."""
    P, V, safe = exact_safe_set(n)
    learned = np.asarray(value_fn(np.stack([P.ravel(), V.ravel()], -1))).reshape(P.shape) >= 0
    iou = (safe & learned).sum() / (safe | learned).sum()
    assert iou >= threshold, f"IoU {iou:.3f} < {threshold}: train longer, raise gamma or check the resets"
    print(f"`test_safe_set_iou` passed: IoU = {iou:.3f}")


# ================================================================ submission check (last cell of the notebook)
def check_submission(*, safety_target, safety_filter, cart_q, part2_failed, car_q, grid, part3_unfiltered,
                     part3_filtered, part3_reached, episodes_needed, part4_task, part4_room, answers):
    """Prints PASS or FAIL for each part, so that a reviewer only has to look at the bottom of the notebook.

    Part 1: safety_target is correct and the cart's learned safe set overlaps the exact one (IoU >= 0.85).
    Part 2: safety_filter is correct and the reckless cart leaves the track in at most 1% of episodes with it.
    Part 3: the car's critic is cautious (precision >= 90%); with the filter the driver crashes in at most 1% of
            at least 1000 episodes, less often than without it, and still reaches the goal in at least 90%.
    Part 4: episodes_needed is correct, both evaluations use at least 10,000 episodes, and the failure rate
            from the start area is below 1% even at the top of its 95% interval.
    Answers: each of the three answers has at least 25 words."""
    count = lambda x: int(x.sum()) if hasattr(x, "sum") else int(x)
    rate = lambda k, n: f"{k}/{n} ({wilson_interval(k, n)[0]:.2%}-{wilson_interval(k, n)[1]:.2%})"
    results, details, problems = {}, {}, {}

    def must_pass(test, function, name):
        try:
            test(function)
        except Exception as e:
            raise AssertionError(f"{name} fails its test ({str(e).strip().splitlines()[0]})") from None

    def part1():
        must_pass(test_safety_target, safety_target, "safety_target")
        P, V, safe = exact_safe_set(101)
        learned = q_value_fn(cart_q)(np.stack([P.ravel(), V.ravel()], -1)).reshape(P.shape) >= 0
        iou = (safe & learned).sum() / (safe | learned).sum()
        details["Part 1 (safety target)"] = f"cart safe set overlap (IoU) {iou:.2f}"
        assert iou >= 0.85, f"IoU {iou:.2f} is below 0.85"

    def part2():
        must_pass(test_safety_filter, safety_filter, "safety_filter")
        k, n = count(part2_failed), len(part2_failed)
        details["Part 2 (safety filter)"] = f"reckless cart with filter: {k}/{n} episodes off the track"
        assert k <= 0.01 * n, (f"{k}/{n} episodes left the track with the filter (allowed: 1%). If safety_target and "
                               "safety_filter pass their tests, retrain the cart's network with another seed")

    def part3():
        with t.inference_mode():
            learned = car_q(car_grid_obs(grid)).max(-1).values >= 0
        reference = grid["V"].flatten() >= 0
        precision = ((learned & reference).sum() / learned.sum().clamp(min=1)).item()
        recall = ((learned & reference).sum() / reference.sum()).item()
        k0, k1, g, n = count(part3_unfiltered), count(part3_filtered), count(part3_reached), len(part3_filtered)
        details["Part 3 (robot car)"] = (f"critic precision {precision:.1%}, recall {recall:.1%}; crashes {k0}/{len(part3_unfiltered)} "
                                         f"without filter, {k1}/{n} with; goal reached {g}/{n}")
        assert precision >= 0.90, f"precision {precision:.0%} is below 90%: the critic calls unsafe states safe"
        assert n >= 1000, f"only {n} episodes; use at least 1000"
        assert k1 < k0 and k1 <= 0.01 * n, (f"{k1}/{n} crashes with the filter (allowed: 1%, and fewer than without). "
                                           "Try another value of car_eps, or retrain the car's network with another seed")
        assert g >= 0.90 * n, f"the filtered driver reached the goal in only {g}/{n} episodes"

    def part4():
        assert episodes_needed(0.01) == 300 and episodes_needed(0.001) == 3000, "episodes_needed is not the rule of three"
        k_task, n_task, k_room, n_room = count(part4_task), len(part4_task), count(part4_room), len(part4_room)
        details["Part 4 (evidence)"] = f"start area {rate(k_task, n_task)}; anywhere in the room {rate(k_room, n_room)}"
        assert min(n_task, n_room) >= 10_000, "use at least 10,000 episodes for each evaluation"
        assert wilson_interval(k_task, n_task)[1] < 0.01, "from the start area, the failure rate could still be 1% or more"

    def short_answers():
        filled = [len(str(a).split()) >= 25 for a in answers.values()]
        details["Short answers"] = f"{sum(filled)}/{len(filled)} answered"
        assert len(filled) == 3 and all(filled), "each of the three answers needs at least 25 words"

    for name, check in [("Part 1 (safety target)", part1), ("Part 2 (safety filter)", part2), ("Part 3 (robot car)", part3),
                        ("Part 4 (evidence)", part4), ("Short answers", short_answers)]:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                check()
            results[name] = True
        except Exception as e:  # a failed check, or a crash in your code
            results[name], problems[name] = False, f"{type(e).__name__}: {str(e).strip().splitlines()[0]}"

    print("=== Submission check ===")
    for name, ok in results.items():
        print(f"{name:<24} {'PASS' if ok else 'FAIL'}   {details.get(name, '')}")
        if not ok:
            print(f"{'':<24}   -> {problems[name]}")
    if all(results.values()):
        print("All parts pass. Hand in the Colab link with outputs saved (Share > Anyone with the link can view).")
    else:
        print("Not complete yet: fix the parts marked FAIL and rerun this cell.")
    return results
'''

Path("part6_safety_filters/safety_starter.py").write_text(STARTER)
print("Starter code written.")

Starter code written.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch as t
from IPython.display import display

from part21_dqn.solutions import DQNArgs, DQNTrainer  # ARENA's DQN, exactly as in section 2.2.1
from part6_safety_filters.safety_starter import (
    FAILURE_MARGIN, DoubleIntegrator, RobotCarTask, car_grid_obs, car_grid_value, check_submission, go_to_goal,
    plot_car_slice, plot_safe_set, q_value_fn, report, rollout, rollout_video, sample_car_starts, sample_safe_states,
    test_safe_set_iou, test_safety_filter, test_safety_target,
)

# Part 1 · Learn which states are safe

### The cart

A cart moves on a frictionless track that runs from −1 m to +1 m. Every 0.05 s it chooses one of three actions: push left, coast, or push right (an acceleration of, respectively, −1, 0 or +1 m/s²). It **fails** when it leaves the track. Its state is $s = (p, v)$: position and velocity.

The **margin** $\ell(s)$ says how far the cart is from failing *right now*: on the track it is the distance to the nearest end, $\ell(s) = 1 - |p|$, and once the cart is off the track it is $-1$.

The margin only describes the present. Take a cart at $p = 0.5$ moving right at $v = 1.5$ m/s. Its margin is positive, but it needs $v^2 / 2a = 1.125$ m to stop and has only 0.5 m left. It has not failed yet, **but** it can no longer avoid failing.

The plot below shows the **exact safe set** in black: all states from which the cart can still stay on the track forever. Everything outside it is doomed, even where the cart is still on the track. For this simple system the safe set can be computed exactly, which gives you something to check your learned result against. The dashed curves are the braking distance $v^2/2a$.

In [ ]:
plot_safe_set()
plt.show()

### The safety value

The **safety value** of a state is the best worst-case margin the cart can achieve from there, if it acts as carefully as possible:

$$V(s) = \max_{\pi} \ \min_{t \ge 0} \ \ell(s_t)$$

If $V(s) \ge 0$, some policy $\pi$ keeps the cart on the track forever. The safe set can be defined as $\{s : V(s) \ge 0\}$.

Like the return in RL, this value obeys a Bellman equation, with a minimum in place of the sum: you are safe if you are fine now *and* you can reach a safe next state. With a discount $\gamma$, which makes it learnable (this is because it turns the entire operator into a *contraction operator*, see [[Fisac et al., 2019]](https://ieeexplore.ieee.org/document/8794107), a necessary condition for Bellman updates to converge to a solution), the target for $Q_{\text{safe}}(s, a)$ is

$$y = (1 - \gamma)\,\ell(s) + \gamma \, \min\{\, \ell(s),\ \max_{a'} Q_{\text{safe}}(s', a') \,\}$$

If the step **terminated** (the cart left the track), there is no next action: use $\ell(s')$ in place of $\max_{a'} Q_{\text{safe}}(s', a')$.

Compare this with the DQN target you know, $y = r + \gamma \max_{a'} Q(s', a')$: the reward is replaced by the margin, and the sum by a minimum.

### Exercise - implement `safety_target`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 10-15 minutes on this exercise.
> ```

All arguments except `gamma` are tensors of shape `(batch,)`. Use `terminated` to choose between `next_q_max` and `next_margin`.

In [ ]:
def safety_target(margin, next_q_max, next_margin, terminated, gamma):
    """The training target y for Q_safe(s, a).

    margin:      l(s), the margin of the current state
    next_q_max:  max_a' Q_safe(s', a'), from the target network
    next_margin: l(s'), the margin of the next state
    terminated:  True where the step ended the episode (bool)
    """
    # YOUR CODE HERE (2-3 lines)
    raise NotImplementedError()


test_safety_target(safety_target)

### Exercise - change one line of the DQN trainer

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 10-15 minutes on this exercise.
> ```

Below is the `training_step` of ARENA's `DQNTrainer`, unchanged except for the line that computes the target `y`. Replace that line with a call to your `safety_target`. Two facts about this project's environments tell you what to pass:

- The **reward** slot of every transition holds $\ell(s)$, the margin of the state the action was taken in. So `data.rewards` is the margin.
- An episode only **terminates** by failing. So whenever `data.terminated` is true, the next state's margin is `FAILURE_MARGIN` (which is −1).

The cell then trains the network on the cart for about half a minute, plots the learned safety value with its zero level set in red, and checks it against the exact safe set.

Two things differ from ordinary DQN training, and both are about coverage. The cart restarts from a random state whenever an episode ends, so that the network sees the whole state space. Further, it spends quite some time during training exploring random actions (`exploration_fraction=0.7` means that the exploration rate will be linearly decayed to its minimum until 70% of training). The filter will be asked about actions a careful policy would never choose, such as pushing towards the end of the track at full speed, so the network has to see those too during training. By having the agent extensively explore both the action and state spaces, we ensure that it will learn (decent) approximations of the safety Q-values, which the safety filter will use to decide when to intervene.

In [ ]:
class SafetyDQNTrainer(DQNTrainer):
    """ARENA's DQNTrainer with one change: the TD target is the safety target."""

    def training_step(self, step: int) -> None:
        data = self.buffer.sample(self.args.batch_size)  # obs, actions, rewards, terminated, next_obs
        with t.inference_mode():
            target_max = self.target_network(data.next_obs).max(-1).values
        predicted_q_vals = self.q_network(data.obs).gather(-1, data.actions.unsqueeze(-1)).squeeze(-1)

        # ARENA's DQN has:  y = data.rewards + self.args.gamma * target_max * (1 - data.terminated.float())
        # YOUR CODE HERE (1 line): compute y with safety_target instead
        raise NotImplementedError()

        loss = (y - predicted_q_vals).pow(2).mean()
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()
        if step % self.args.trains_per_target_update == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())


cart_args = DQNArgs(env_id="DoubleIntegrator-v0", device="cpu", total_timesteps=150_000, learning_rate=1e-3,
                    exploration_fraction=0.7, end_e=0.2, steps_per_live_video=None
                    #, seed=1
                    ) # If the filter does not prevent the reckless controller (see below) from crashing, uncomment the seed=1 line and retrain the agent
cart_trainer = SafetyDQNTrainer(cart_args)
cart_trainer.train()
cart_q = cart_trainer.q_network

plot_safe_set(q_value_fn(cart_q))
plt.show()
test_safe_set_iou(q_value_fn(cart_q))

Look at where the red boundary (learned) and the black one (exact) differ. You will need this for the first short answer at the end.

# Part 2 · Build the filter

A **least-restrictive safety filter** sits between a controller and the robot. At every step the controller proposes an action $a_{\text{nom}}$ (the *nominal* action), and the filter decides:

- if $Q_{\text{safe}}(s, a_{\text{nom}}) \ge \varepsilon$, the proposed action is safe enough, so let it through;
- otherwise, override it with the safest action, $\arg\max_a Q_{\text{safe}}(s, a)$.

The threshold $\varepsilon \ge 0$ is a buffer: the larger it is, the earlier the filter steps in.

### Exercise - implement `safety_filter`

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 10 minutes on this exercise.
> ```

In [ ]:
def safety_filter(q_values, a_nom, eps):
    """q_values: Q_safe(s, .), shape (batch, n_actions). a_nom: the controller's actions, shape (batch,).
    Returns the actions to execute, shape (batch,)."""
    # YOUR CODE HERE (2-3 lines)
    raise NotImplementedError()


test_safety_filter(safety_filter)

### A reckless controller

Now test the filter on the worst controller imaginable: it always pushes right. The cell runs 500 carts at once, each starting from a different state inside the exact safe set, first without and then with your filter.

`rollout` (from the starter code) runs one episode per cart and returns a dictionary; `result["failed"]` holds one boolean per cart. The plot shows three of the filtered carts; orange dots mark the steps where the filter overrode the controller.

In [ ]:
cart_starts = sample_safe_states(500)  # 500 states drawn from the exact safe set
cart_env = DoubleIntegrator(num_envs=500)
reckless = lambda obs: t.full((len(obs),), 2)  # action 2 = push right, always

unfiltered = rollout(cart_env, reckless, init_states=cart_starts)
filtered = rollout(cart_env, reckless, cart_q, eps=0.05, init_states=cart_starts, safety_filter=safety_filter, record=True)
part2_failed = filtered["failed"]

report("Reckless cart, no filter", unfiltered["failed"])
report("Reckless cart, your filter", part2_failed, f"; the filter overrides {filtered['intervention_rate']:.0%} of steps")

plot_safe_set(q_value_fn(cart_q), [(filtered["states"][:, i], filtered["interventions"][:, i]) for i in range(3)])
plt.show()

If your filter fails the test but the functions you wrote in the previous two exercise cells pass theirs, the training run was unlucky: add `seed=1` to `DQNArgs` in Part 1, retrain, and run this cell again.

Once a cart reaches the boundary of the learned safe set, the filter overrides the controller on most steps: the filter brakes, the controller pushes again on the next step, the filter brakes again, and so on. This back-and-forth along the boundary is called *chattering*. With a controller this stubborn, the cart ends up parked near the end of the track, held there by the filter.

### Exercise - how much buffer?

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to 10 minutes on this exercise.
> ```

The reckless controller is an extreme case. A controller that acts **randomly** is more typical of something that simply is not paying attention. Complete the loop so that, for each $\varepsilon$, it runs the random controller through your filter from `cart_starts` and stores two numbers: how many episodes failed, and on what fraction of steps the filter overrode the controller.

In [ ]:
def random_policy(n_actions, seed=0):
    generator = t.Generator().manual_seed(seed)
    return lambda obs: t.randint(0, n_actions, (len(obs),), generator=generator)


cart_sweep = {}  # eps -> (number of failed episodes out of 500, fraction of steps the filter overrode)
for eps in [0.0, 0.05, 0.1, 0.2, 0.4]:
    # YOUR CODE HERE (2 lines): use rollout with random_policy(3) and your filter, then fill cart_sweep[eps].
    # A result has "failed" (one boolean per cart) and "intervention_rate" (a float).
    raise NotImplementedError()

report("Random cart, no filter", rollout(cart_env, random_policy(3), init_states=cart_starts)["failed"])
for eps, (k, rate) in cart_sweep.items():
    print(f"  your filter, eps = {eps}: {k} of 500 episodes fail; the filter overrides {rate:.0%} of steps")

Look at both columns. A larger buffer costs freedom, because the filter overrides more often. Whether it also buys safety depends on how accurate the critic is. Keep this in mind for the car in Part 3, where the critic is less accurate than here.

# Part 3 · A robot car in a room

The cart was a warm-up with a known answer. Now the same method meets a system where the safe set is no longer a formula.

A car drives through a 2 m × 2 m room with three round obstacles. It moves at a constant 0.5 m/s and cannot brake: every 0.05 s it chooses to **steer right, go straight or steer left**, with a turning radius of 0.25 m. It fails when it touches a wall or an obstacle. Its state is its position and heading (represented as an angle $\theta$), meaning that the network observes $(x, y, \cos\theta, \sin\theta)$.

The car's task is to drive from the bottom-left corner to the green circle. The driver below, `go_to_goal`, steers straight at the goal and ignores everything else.

The video shows 16 cars at once. **Blue**: the driver is in charge. **Red**: crashed. **Green**: reached the goal. Later you will also see **orange**: the filter is overriding the driver.

In [ ]:
demo_env = RobotCarTask(num_envs=16)
demo = rollout(demo_env, go_to_goal, record=True)
report("16 cars, driver ignores obstacles", demo["failed"])
display(rollout_video(demo_env, demo))

### What is actually safe? A grid solver

The car's state has three dimensions, which is few enough to compute the safe set almost exactly by brute force: put a grid over $(x, y, \theta)$ and repeat the backup $V(s) \leftarrow \min\{\ell(s), \max_a V(s')\}$ until nothing changes. This is the *reachability* computation from the lecture. It is exact up to the grid resolution, and it is the reference you will compare your learned critic against. It takes less than a minute.

Each plot shows the room for one heading of the car (the arrow in the top-left corner). Inside the black line the car is safe. Notice the "shadow" on the side of each obstacle that the car is driving towards: there the car has not hit anything yet, but it can no longer turn away in time.

In [ ]:
grid = car_grid_value()  # less than a minute

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
for ax, heading in zip(axes, (0, 45, 135)):
    plot_car_slice(heading, grid=grid, ax=ax)
plt.show()

### Train a safety critic for the car

Nothing changes in the method: the same `SafetyDQNTrainer`, pointed at the car (though one training detail does change: here we set `exploration_fraction=0.2`, so that the exploration rate linearly decays to its minimum until 20% of training.) The margin is now the car's clearance (its distance to the nearest wall or obstacle, rescaled so that 0.25 m or more counts as 1), and −1 after a collision. Training takes about a minute; training episodes are short and restart anywhere in the room, so that the network sees every corner of it.

The plots add your critic to the picture: the colours show its value (blue is positive), the red line is where it crosses zero, and the black line is still the reference.

In [ ]:
car_args = DQNArgs(env_id="RobotCar-v0", device="cpu", num_envs=64, total_timesteps=1_000_000, buffer_size=100_000,
                   learning_rate=1e-3, exploration_fraction=0.2, steps_per_live_video=None)
car_trainer = SafetyDQNTrainer(car_args)
car_trainer.train()
car_q = car_trainer.q_network

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), constrained_layout=True)
for ax, heading in zip(axes, (0, 45, 135)):
    plot_car_slice(heading, grid=grid, q_net=car_q, ax=ax)
plt.show()

### Exercise - is your critic cautious or reckless?

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 10 minutes on this exercise.
> ```

The red line (learned) and the black line (reference) do not coincide. Two numbers say in which direction your critic errs:

- **precision**: of the states your critic calls safe, what fraction does the reference also call safe? Low precision means the critic lets the car into states that are in fact doomed.
- **recall**: of the states the reference calls safe, what fraction does your critic call safe? Low recall means the critic is over-cautious and forbids states that are fine.

`learned_safe` and `reference_safe` are boolean tensors with one entry per grid point. Compute both numbers as Python floats.

In [ ]:
with t.inference_mode():
    learned_safe = car_q(car_grid_obs(grid)).max(-1).values >= 0
reference_safe = grid["V"].flatten() >= 0

# YOUR CODE HERE (2 lines)
car_precision = ...
car_recall = ...
print(f"precision {car_precision:.1%}, recall {car_recall:.1%}")

### Exercise - choose the buffer for the car

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 15 minutes on this exercise.
> ```

Run 1000 cars from the start area, each driven by `go_to_goal` through your filter, for several values of $\varepsilon$. For each one, count how many cars crash and how many reach the goal. Then set `car_eps` to the smallest value in your sweep for which no car crashes and at least 95% of the cars still reach the goal. If no value gets to zero crashes, take the one with the fewest. A buffer that is too large is safe but useless: the filter then blocks the way and the cars never arrive.

The cell then reruns that choice, records it, and shows the video.

In [ ]:
car_env = RobotCarTask(num_envs=1000)
car_starts = sample_car_starts(1000, grid)  # starts in the bottom-left area that the reference calls safe
part3_unfiltered = rollout(car_env, go_to_goal, init_states=car_starts)["failed"]
report("Car, no filter", part3_unfiltered)

car_sweep = {}  # eps -> (number of crashes, number of cars that reached the goal), out of 1000
for eps in [0.0, 0.1, 0.2, 0.3, 0.5, 0.7]:
    # YOUR CODE HERE (2 lines): use rollout with go_to_goal and your filter; a result has "failed" and "reached_goal"
    raise NotImplementedError()
for eps, (crashes, goals) in car_sweep.items():
    print(f"  your filter, eps = {eps}: {crashes} crash, {goals} reach the goal")

car_eps = ...  # YOUR CHOICE

result = rollout(car_env, go_to_goal, car_q, car_eps, init_states=car_starts, safety_filter=safety_filter, record=True)
part3_filtered, part3_reached = result["failed"], result["reached_goal"]
report(f"Car, your filter with eps = {car_eps}", part3_filtered, f"; the filter overrides {result['intervention_rate']:.0%} of steps")
display(rollout_video(car_env, result))

# Part 4 · How strong is your evidence?

Suppose your filtered car did not crash in any of the 1000 episodes. Can you say it never crashes? No: a car that crashes once in 2000 episodes would usually also show zero crashes in 1000.

The **rule of three** turns "no failures so far" into a claim you can defend. If you see zero failures in $n$ independent episodes, then with 95% confidence the true failure rate is below about $3/n$. (A failure rate $p$ gives zero failures in $n$ episodes with probability $(1-p)^n \approx e^{-np}$, and this drops to 5% when $np \approx 3$.)

### Exercise - how many episodes do you need?

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to 5 minutes on this exercise.
> ```

Write the rule of three the other way round: how many episodes in a row without a failure do you need, before you can claim that the failure rate is below `max_failure_rate`? Return a whole number of episodes.

In [ ]:
def episodes_needed(max_failure_rate):
    # YOUR CODE HERE (1 line)
    raise NotImplementedError()


assert episodes_needed(0.01) == 300 and episodes_needed(0.001) == 3000
print("To claim a failure rate below 0.03%, you need", episodes_needed(0.0003), "episodes without a failure.")

### Evidence at scale

Simulated cars are cheap, so we can run 10,000 of them 😃. The cell below does this twice, with your filter and your `car_eps`:

1. from the **start area** in the bottom-left corner, as in Part 3;
2. from safe states **anywhere in the room**, with any heading. "Safe" here means that the grid reference gives them at least 5 cm to spare.

`report` prints each failure count with its 95% interval.

In [ ]:
N = 10_000
big_env = RobotCarTask(num_envs=N)
task_starts = sample_car_starts(N, grid, seed=1)              # the start area
room_starts = sample_car_starts(N, grid, seed=2, task=False)  # anywhere in the room, any heading

part4_task = rollout(big_env, go_to_goal, car_q, car_eps, init_states=task_starts, safety_filter=safety_filter)["failed"]
part4_room = rollout(big_env, go_to_goal, car_q, car_eps, init_states=room_starts, safety_filter=safety_filter)["failed"]
report("Filtered car, from the start area", part4_task)
report("Filtered car, from anywhere in the room", part4_room)

The two numbers answer two very different questions. The first says how the filtered car behaves on the task it was tested on. The second asks the filter to cope with every state the reference calls "safe", including states right at the edge of the safe set, where the car needs precise control and the learned critic is least accurate.

# Three short answers

Answer each question in 2 to 4 sentences, inside the triple quotes, by referring to your own plots and numbers. The answers are checked for completeness but not graded, so think of them as a reflection moment to help you understand what you have learned during the lecture and this (short) project.

In [ ]:
answers = {
    "1. Where does your learned safe set for the cart disagree with the exact one (Part 1 plot), and what could explain it?": """

    """,
    "2. Your filter is learned, not computed exactly. What does it actually guarantee? Use your Part 3 and Part 4 numbers.": """

    """,
    "3. How could a robot notice at run time that its safety value is wrong? Name one signal you could compute in this notebook.": """

    """,
}

# Submission check

Run this last. It tests every part and prints `PASS` or `FAIL` for each; when all five lines say `PASS`, save the notebook with its outputs and hand in the link. Well done!

In [ ]:
submission = check_submission(
    safety_target=safety_target, safety_filter=safety_filter,
    cart_q=cart_q, part2_failed=part2_failed,
    car_q=car_q, grid=grid, part3_unfiltered=part3_unfiltered, part3_filtered=part3_filtered, part3_reached=part3_reached,
    episodes_needed=episodes_needed, part4_task=part4_task, part4_room=part4_room,
    answers=answers,
)